# Static Gold Fact

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from delta.tables import DeltaTable

In [0]:
df_fact = spark.sql("""

WITH airports AS 
(
    SELECT * FROM flyproject.gold.dim_airports
),

flights AS 
(
    SELECT * FROM flyproject.gold.dim_flights
),

passengers AS
(
    SELECT * FROM flyproject.gold.dim_passengers
),

final_query AS
(
SELECT
    fact.booking_id,
    fact.booking_date,
    fact.amount,
    airports.DimAirportsKey,
    flights.DimFlightsKey,
    passengers.DimPassengersKey
FROM flyproject.silver.silver_bookings AS fact
LEFT JOIN airports
    ON fact.airport_id = airports.airport_id
LEFT JOIN flights
    ON fact.flight_id = flights.flight_id
LEFT JOIN passengers
    ON fact.passenger_id = passengers.passenger_id
)

SELECT DISTINCT
    *
FROM final_query

""")

In [0]:
display(df_fact)

In [0]:
if spark.catalog.tableExists("flyproject.gold.fact_bookings"):
    target_table = DeltaTable.forName(spark, "flyproject.gold.fact_bookings")
    target_table.alias("target") \
                .merge(df_fact.alias("source"), "target.booking_id = source.booking_id") \
                .whenMatchedUpdateAll() \
                .whenNotMatchedInsertAll() \
                .execute()

else:
    # initial load
    df_fact.write \
           .mode("overwrite") \
           .format("delta") \
           .saveAsTable("flyproject.gold.fact_bookings")

In [0]:
%sql
SELECT 
    booking_id,
    COUNT(*) 
FROM flyproject.gold.fact_bookings
GROUP BY booking_id
HAVING COUNT(*) > 1